# Step 2 — Feature engineering and non-learned baselines

Builds ~105 features for every displayed (user, item) pair and evaluates simple one-signal baselines on the same full slates every model is evaluated on.

### Setup (run first)

**On Google Colab:** *Runtime → Change runtime type → GPU* (T4 is enough; A100/L4 is faster). Put the two raw JDsearch
files in `MyDrive/bt4222/raw/` (`user_behavior_data.txt`, `product_meta_data.txt`). The cell mounts Drive, clones the
repository, installs the requirements and writes every output to `MyDrive/bt4222/outputs/`, so results persist between
sessions and the notebooks can be run one after another (or on different days).

**Locally:** run the notebook from the repository with the virtual environment active; data is read from `data/raw/`
and outputs go to `outputs/`.

`CONFIG = "configs/smoke.yaml"` runs everything on a 2,000-user sample with tiny models (minutes, CPU) to check the code;
`configs/default.yaml` is the real run. Every stage skips work whose outputs already exist; pass `"--force"` to redo one.

In [ ]:
import os, subprocess, sys
from pathlib import Path

CONFIG = "configs/default.yaml"          # "configs/smoke.yaml" for a quick check
REPO_URL = "https://github.com/clarequek/bt4222-two-tower-recsys.git"
DRIVE_DIR = "/content/drive/MyDrive/bt4222"   # Colab: raw/ (input) and outputs/ (written by the pipeline)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/bt4222-two-tower-recsys")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from jds.notebook import Runner, banner
run = Runner(CONFIG, drive_dir=DRIVE_DIR if IN_COLAB else None)

## 2.1 Features (`scripts/02_features.py`, `src/jds/features.py`)

Each data field has one role:

| data field | role |
|---|---|
| test query + displayed slate + labels | the supervised task (query-conditioned re-ranking of the slate) |
| the user's own history | personal features, **counted per event type inside that type's retention window** (`click_7`, `cart_90`, `order_365`, …) plus days since the last event |
| query-tagged history of all users | query → item / brand / category engagement (`qpop_*`) |
| history of all users | windowed item / attribute popularity (`ipop_*`, `apop_*`) |
| product metadata | BM25 and query-term coverage (`lex_*`), item content — needed because most displayed items are cold |
| labels of *other* training users | impression CTR (`ctr_*`), **out-of-fold**: a training user never sees its own labels |

Also: engagement under the user's earlier queries similar to the test query (`u_q_*`, `u_simq_*`), carted-but-not-ordered
(`u_cart_not_ordered`), slate context and within-slate percentiles (`slate_*`, `*_pct`).
Peak RAM is ~6 GB; on a small Colab runtime add `"--set", "features.chunk_users=10000"`.

In [ ]:
run.stage("02_features.py")

In [ ]:
import json
names = json.loads((run.p.features / "feature_list.json").read_text())
print(len(names), "features"); names[:20]

## 2.2 Baselines (`scripts/02b_baselines.py`, `src/jds/baselines.py`)

Each baseline ranks a slate by one signal (random, windowed popularity, CTR, query–item engagement, BM25, user affinity),
plus a popularity + BM25 mix whose weight is picked on validation users. Metrics: MRR, P@1, NDCG@10 (binary, as in the
JDsearch paper), graded NDCG@10 (gain 2^label − 1), HR@10, recall@10/50.

In [ ]:
run.stage("02b_baselines.py")

In [ ]:
banner()
import pandas as pd
b = pd.DataFrame(run.json("baselines.json"))
b = b[b.model.str.endswith("[test]")]
b.style.format(precision=4).highlight_max(subset=["mrr", "p1", "ndcg10", "gndcg10", "hr10"], color="#c6efce")